In [10]:
# !pip install -q openpyxl tqdm

from openpyxl import load_workbook
from tqdm.auto import tqdm
import re
import html

# =========================
# 1. File paths
# =========================
MAIN_FILE = "/content/franckenachlass_2.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"
CORRECTIONS_FILE = "/content/Wikidata_place_updated.xlsx"
OUTPUT_FILE = "/content/franckenachlass_3.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"

HEADER_ROW = 1

TARGET_COLUMNS = [
    "creator_linked_new",
    "receiver_linked_new",
    "sender_place_linked_new",
    "receiver_place_linked_new"
]

# Correction file columns
CORR_LINK_COL = "LODwikiData"
CORR_QID_COL = "Q_ID"
CORR_VALUE_COL = "wikidata_label_final"

# =========================
# 2. Regex
# =========================
# Accept both browser pages (/wiki/QID) and canonical entity URIs (/entity/QID).
WIKIDATA_URL_RE = re.compile(
    r"https?://(?:www\.)?wikidata\.org/(?:wiki|entity)/(Q\d+)",
    re.IGNORECASE
)

WIKIDATA_ANCHOR_RE = re.compile(
    r'(?P<start><a\b[^>]*\bhref=["\'](?P<href>https?://(?:www\.)?wikidata\.org/(?:wiki|entity)/(?P<qid>Q\d+)[^"\']*)["\'][^>]*>)'
    r'(?P<label>.*?)'
    r'(?P<end></a>)',
    re.IGNORECASE | re.DOTALL
)


def extract_qid(text):
    if text is None:
        return None

    text = str(text)

    m = WIKIDATA_URL_RE.search(text)
    if m:
        return m.group(1).upper()

    m = re.search(r"\bQ\d+\b", text, re.IGNORECASE)
    if m:
        return m.group(0).upper()

    return None


def normalize_wikidata_uri(value):
    """
    Use the canonical RWO/entity URI, not the browser /wiki/ URL.
    Example: http://www.wikidata.org/entity/Q19526
    """
    qid = extract_qid(value)
    if qid:
        return f"http://www.wikidata.org/entity/{qid}"
    return str(value).strip() if value is not None else ""


def format_wikidata_label(label, qid):
    """
    Display the corrected Wikidata label and include the QID once.
    Example: George Fox (Q19526)
    """
    label = str(label).strip()
    if qid and qid not in label:
        return f"{label} ({qid})"
    return label


def replace_href_in_anchor(start_tag, new_href):
    escaped_href = html.escape(new_href, quote=True)
    return re.sub(
        r'href=["\'][^"\']+["\']',
        f'href="{escaped_href}"',
        start_tag,
        count=1,
        flags=re.IGNORECASE
    )


def get_header_map(ws):
    header_map = {}
    for col in range(1, ws.max_column + 1):
        value = ws.cell(HEADER_ROW, col).value
        if value is not None:
            header_map[str(value).strip()] = col
    return header_map


# =========================
# 3. Load corrected Wikidata labels
# =========================
corr_wb = load_workbook(CORRECTIONS_FILE, data_only=True)
corr_ws = corr_wb.active

corr_headers = get_header_map(corr_ws)

required_corr_cols = [CORR_LINK_COL, CORR_QID_COL, CORR_VALUE_COL]
missing_corr_cols = [c for c in required_corr_cols if c not in corr_headers]

if missing_corr_cols:
    raise ValueError(f"Missing columns in correction file: {missing_corr_cols}")

corr_link_idx = corr_headers[CORR_LINK_COL]
corr_qid_idx = corr_headers[CORR_QID_COL]
corr_value_idx = corr_headers[CORR_VALUE_COL]

wikidata_label_by_qid = {}

for row in range(HEADER_ROW + 1, corr_ws.max_row + 1):
    link_value = corr_ws.cell(row, corr_link_idx).value
    qid_value = corr_ws.cell(row, corr_qid_idx).value
    final_label = corr_ws.cell(row, corr_value_idx).value

    if final_label is None or str(final_label).strip() == "":
        continue

    qid = extract_qid(qid_value) or extract_qid(link_value)

    if qid:
        wikidata_label_by_qid[qid] = str(final_label).strip()

print(f"Loaded {len(wikidata_label_by_qid)} corrected Wikidata labels.")


# =========================
# 4. Replace Wikidata anchor labels and normalize URI
# =========================
def replace_wikidata_label_in_cell(cell_value):
    if cell_value is None or not isinstance(cell_value, str):
        return cell_value, []

    lower_value = cell_value.lower()
    if "wikidata.org/wiki/" not in lower_value and "wikidata.org/entity/" not in lower_value:
        return cell_value, []

    changes = []

    def repl(match):
        qid = match.group("qid").upper()
        old_href = match.group("href")
        old_label = match.group("label")

        corrected_label = wikidata_label_by_qid.get(qid)
        new_href = normalize_wikidata_uri(qid)

        # If the QID is not in the correction file, still normalize the URI.
        if corrected_label:
            new_label = format_wikidata_label(corrected_label, qid)
        else:
            new_label = old_label

        escaped_new_label = html.escape(new_label, quote=False)
        new_start = replace_href_in_anchor(match.group("start"), new_href)

        if old_href == new_href and old_label == escaped_new_label:
            return match.group(0)

        changes.append({
            "qid": qid,
            "old_href": old_href,
            "new_href": new_href,
            "old_label": old_label,
            "new_label": new_label
        })

        return f"{new_start}{escaped_new_label}{match.group('end')}"

    new_value = WIKIDATA_ANCHOR_RE.sub(repl, cell_value)

    return new_value, changes


# =========================
# 5. Load main Excel file
# =========================
wb = load_workbook(MAIN_FILE)
ws = wb.active

main_headers = get_header_map(ws)

missing_main_cols = [c for c in TARGET_COLUMNS if c not in main_headers]
if missing_main_cols:
    raise ValueError(f"Missing columns in main file: {missing_main_cols}")

log_rows = []
total_replacements = 0

for col_name in TARGET_COLUMNS:
    col_idx = main_headers[col_name]

    for row in tqdm(
        range(HEADER_ROW + 1, ws.max_row + 1),
        desc=f"Processing {col_name}"
    ):
        cell = ws.cell(row, col_idx)
        old_value = cell.value

        new_value, changes = replace_wikidata_label_in_cell(old_value)

        if changes:
            cell.value = new_value
            total_replacements += len(changes)

            for ch in changes:
                log_rows.append([
                    row,
                    col_name,
                    ch["qid"],
                    ch["old_href"],
                    ch["new_href"],
                    ch["old_label"],
                    ch["new_label"]
                ])

print(f"Total Wikidata labels or URIs updated: {total_replacements}")


# =========================
# 6. Add log sheet
# =========================
LOG_SHEET = "wikidata_replacement_log"

if LOG_SHEET in wb.sheetnames:
    del wb[LOG_SHEET]

log_ws = wb.create_sheet(LOG_SHEET)

log_ws.append([
    "row",
    "column",
    "qid",
    "old_wikidata_href",
    "new_wikidata_href",
    "old_wikidata_label",
    "new_wikidata_label"
])

for item in log_rows:
    log_ws.append(item)


# =========================
# 7. Save output
# =========================
wb.save(OUTPUT_FILE)

print("Done.")
print("Saved file:", OUTPUT_FILE)


Loaded 505 corrected Wikidata labels.


Processing creator_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Processing receiver_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Processing sender_place_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Processing receiver_place_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Total Wikidata labels or URIs updated: 0
Done.
Saved file: /content/franckenachlass_3.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx
